# Traffic Intelligence

In [ ]:
!pip uninstall -y ultralytics torch torchvision torchaudio

!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

!pip install ultralytics==8.3.40 \
opencv-python-headless==4.10.0.84 \
websockets==15.0.1 \
yt-dlp==2024.8.6

Found existing installation: ultralytics 8.3.40
Uninstalling ultralytics-8.3.40:
  Successfully uninstalled ultralytics-8.3.40
Found existing installation: torch 2.12.0
Uninstalling torch-2.12.0:
  Successfully uninstalled torch-2.12.0
Found existing installation: torchvision 0.27.0
Uninstalling torchvision-0.27.0:
  Successfully uninstalled torchvision-0.27.0
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
  Using cached https://download-r2.pytorch.org/whl/triton-3.1.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (209.6 MB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.13

In [ ]:
import asyncio
import base64
import json
import time
from datetime import datetime

import cv2
import torch
import websockets
from ultralytics import YOLO

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda:0


In [ ]:
VIDEO_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4"
VIDEO_PATH = "traffic.mp4"
!wget -q -O {VIDEO_PATH} {VIDEO_URL}
print("Video saved to", VIDEO_PATH)

Video saved to https://storage.googleapis.com/gtv-videos-bucket/sample/BigBuckBunny.mp4


In [ ]:
WS_INGEST_URL = "wss://<your-domain>/ws/ingest"

In [ ]:
CLASSES = {"car", "truck", "bus", "motorcycle", "bicycle", "person"}

model = YOLO("yolov8n.pt")
model.to(device)

def build_counts():
    return {"car": 0, "truck": 0, "bus": 0, "motorcycle": 0, "bicycle": 0, "person": 0}

async def stream_video():
    cap = cv2.VideoCapture(VIDEO_PATH)
    peak_density = 0
    last_frame_time = time.time()
    last_event_at = {"truck": 0, "bus": 0, "spike": 0}

    async with websockets.connect(WS_INGEST_URL, max_size=2**23) as ws:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue

            results = model.predict(source=frame, device=device, conf=0.35, verbose=False)
            result = results[0]

            counts = build_counts()
            for cls_id in result.boxes.cls.tolist():
                label = result.names[int(cls_id)]
                if label in counts:
                    counts[label] += 1

            total = sum(counts.values())
            peak_density = max(peak_density, total)

            now = time.time()
            fps = 1.0 / max(now - last_frame_time, 1e-6)
            last_frame_time = now

            annotated = result.plot()
            cv2.putText(
                annotated,
                f"FPS {fps:.1f}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 255),
                2,
            )

            _, buffer = cv2.imencode(".jpg", annotated, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
            frame_b64 = base64.b64encode(buffer).decode("utf-8")

            events = []
            ts = datetime.utcnow().isoformat()
            if counts["truck"] > 0 and now - last_event_at["truck"] > 3:
                events.append({"message": "Truck detected", "severity": "info", "ts": ts})
                last_event_at["truck"] = now
            if counts["bus"] > 0 and now - last_event_at["bus"] > 3:
                events.append({"message": "Bus entered frame", "severity": "info", "ts": ts})
                last_event_at["bus"] = now
            if peak_density > 0 and total / peak_density >= 0.85 and now - last_event_at["spike"] > 5:
                events.append({"message": "Traffic spike detected", "severity": "critical", "ts": ts})
                last_event_at["spike"] = now

            payload = {
                "timestamp": ts,
                "counts": counts,
                "totalVehicles": total,
                "peakDensity": peak_density,
                "fps": fps,
                "events": events,
                "frame": frame_b64,
            }

            await ws.send(json.dumps(payload))
            await asyncio.sleep(0.03)

    cap.release()

await stream_video()